# 27 · Suy luận lại B3-full với OCR cache đã sửa EXIF (W14, bước 2 — phần B, L4)

**Vấn đề:** Notebook 25 xác nhận 126 câu full test ViTextVQA (1,256%) có OCR rác do cờ xoay EXIF. Luật R-EXIF **L2** bắt buộc: suy luận lại **B3-full** trên **full test** với cache đã sửa (notebook 26) và **báo cả hai số**. **L3** (1,28% câu train ≤ 5%): **không** train lại → số chính thức ViTextVQA vẫn là B3-full W10; bản mới là dòng "độ nhạy".

**Notebook này làm (luật ghi trước ở `docs/W14/README.md`, mục "R-EXIF bước 2"):**
1. Kiểm: biên bản cache đã sửa hợp lệ và trùng sha256 file trên Drive; adapter B3-full; 2 run tham chiếu (B3-full W10, B2 W07) đủ 10.028 câu.
2. **Eval B3-full** full test 10.028 câu với cache đã sửa — cùng adapter, cùng cấu hình OCR (≈ 2,8 h, ghi mỗi 500 câu, bị ngắt thì Run all lại để chạy tiếp).
3. **Chỉ mô tả:** 126 câu ảnh cờ xoay với **ảnh đã xoay** + OCR đã sửa (giống đường ảnh của prototype), ≈ 3 phút.
4. So ghép cặp: kiểm **V5** (câu không bị ảnh hưởng phải có dự đoán trùng run W10 ≥ 99,5%), ΔEM / ΔANLS + McNemar + bootstrap.

**Runtime:** **L4** (cùng GPU với B3-full W10). Tổng ≈ 3 h, cuối notebook tự trả máy. Có thể để chạy qua đêm (máy không ngủ, tab mở).

**Anh chỉ cần:** Run all → gửi Claude **output mục 7**.

## 1. Thiết lập môi trường (như notebook 18: `transformers==5.17.0`, L4)

*Kỳ vọng: commit mới nhất, `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, json, subprocess
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert 'R-EXIF bước 2' in open('docs/W14/README.md', encoding='utf-8').read(), '❌ Luật bước 2 chưa được commit -> DỪNG'
assert '--image_dir' in open('src/eval/run_baseline.py', encoding='utf-8').read(), '❌ Code cũ -> git pull'

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B3-full W10). Runtime -> Change runtime type -> L4, rồi Run all.')
print('✅ Thiết lập xong')

## 2. Kiểm tra trước khi tốn ~3 giờ GPU

*Kỳ vọng: `cache đã sửa hợp lệ ✅ (sha256 trùng biên bản)`, `paddledet_vietocr | v1 | 600 | 0.0`, `adapter B3-full ✅`, 2 dòng `… 10028 câu ✅`, `✅ selftest OK`.*

In [ ]:
from src.data.exif_audit import sha256_file
DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vitextvqa_official'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
AUDIT = f'{EXP_ROOT}/W14_exif_audit_vitextvqa'
OUT = f'{EXP_ROOT}/W14_exif_fix'
FULL_TEST = f'{DRIVE_ROOT}/data/{DS}/test.json'
NEW_CACHE = f'{DRIVE_ROOT}/data/{DS}/ocr/paddledet_vietocr_exiffix.json'
CHUNK = 500

MAN = json.load(open(f'{OUT}/ocr_cache_exiffix_manifest.json', encoding='utf-8'))
assert MAN['valid'], '❌ Biên bản notebook 26 ghi KHÔNG HỢP LỆ -> DỪNG'
assert sha256_file(NEW_CACHE) == MAN['sha256'], '❌ Cache đã sửa trên Drive khác biên bản notebook 26 -> DỪNG'
print('cache đã sửa hợp lệ ✅ (sha256 trùng biên bản)', MAN['sha256'][:12])

sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--mode ocr --ocr_cache {NEW_CACHE} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'

B3F_TRAIN = 'W10_B3full_qlora_ocr_vietocr_v1_r16_qwen25vl_vitextvqa_full_ep1_bf16_L4_seed42'
AD_B3F = f'{DRIVE_ROOT}/adapters/{B3F_TRAIN}/epoch_1'
assert os.path.isdir(AD_B3F), f'❌ Thiếu adapter {AD_B3F}'
print('adapter B3-full ✅')
REF = {'B3full_W10': 'W10_B3full_eval_qlora_ocr_vietocr_v1_r16_full_ep1_vitextvqa_testfull_bf16_L4_seed42',
       'B2_W07': 'W07_B2_eval_qlora_r16_ep1_vitextvqa_testfull_bf16_L4_seed42'}
for k, r in REF.items():
    n = len(json.load(open(f'{EXP_ROOT}/{r}/predictions.json', encoding='utf-8')))
    assert n == 10028, (k, n)
    print(f'{k:11s} {r[:44]}… {n} câu ✅')
NEW_EVAL = 'W14_B3full_eval_exiffix_qlora_ocr_vietocr_v1_r16_full_ep1_vitextvqa_testfull_bf16_L4_seed42'
ROT_EVAL = 'W14_B3full_eval_exiffix_rotimg_qlora_ocr_vietocr_v1_r16_full_ep1_vitextvqa_testexif_bf16_L4_seed42'

def run_cmd(cmd):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}) -> xem log ở trên, gửi Claude')

for mod, must in {'src.eval.run_baseline': 'SELFTEST OK', 'src.data.exif_audit': 'exif_audit SELFTEST OK'}.items():
    o = subprocess.run(f'python -W ignore -m {mod} --selftest', shell=True, capture_output=True, text=True)
    assert must in o.stdout, (mod, o.stdout[-800:], o.stderr[-800:])
print('✅ selftest OK')

## 3. Ảnh ViTextVQA về máy Colab

*Kỳ vọng: dòng thư mục ảnh. Lần đầu khoảng 10 phút (giải nén zip 5,6 GB).*

In [ ]:
from src.eval.run_baseline import prepare_images
IMG_TV = prepare_images(DS, DRIVE_ROOT)
print('ảnh ViTextVQA:', IMG_TV)

## 4. Eval B3-full trên full test với cache đã sửa (≈ 2,8 giờ)

Cùng adapter, cùng cấu hình OCR (VietOCR · v1 · 600 · 0), chỉ đổi file cache. Ghi lên Drive mỗi 500 câu; bị ngắt thì mở lại notebook, Run all → chạy tiếp từ lô dở.

*Kỳ vọng: log tiến độ từng lô, cuối cùng `✔ B3-full (cache sửa EXIF): EM=… ANLS=…` (W10: EM 0.5335 ANLS 0.7389).*

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{NEW_EVAL}/metrics.json'):
    print('⏭ eval đã có -> bỏ qua')
else:
    run_cmd(f'python -u -m src.eval.run_baseline --dataset {DS} --data_file {FULL_TEST} --run_name {NEW_EVAL} '
            f'--chunk {CHUNK} --adapter {AD_B3F} {OCR_ARGS}')
m = json.load(open(f'{EXP_ROOT}/{NEW_EVAL}/metrics.json', encoding='utf-8'))
print(f"✔ B3-full (cache sửa EXIF): EM={m['metrics']['exact_match']:.4f} ANLS={m['metrics']['anls']:.4f} "
      f"F1={m['metrics']['token_f1']:.4f}  (W10: EM 0.5335 ANLS 0.7389)")

## 5. Chỉ mô tả — ảnh đã xoay + OCR đã sửa (126 câu, ≈ 3 phút)

Tạo file data con (câu test có ảnh cờ xoay) và bản ảnh đã xoay (giữ tên file), rồi eval với `--image_dir` trỏ vào thư mục ảnh xoay. Đây là đường ảnh của prototype (giao diện xoay ảnh theo EXIF trước khi đưa vào model).

*Kỳ vọng: dòng JSON `n_questions` ≈ 126, rồi `✔ ảnh xoay: EM=…`.*

In [ ]:
ROT_DIR = '/content/vitextvqa_exif_rotated'
!python -W ignore -m src.data.exif_audit --make_subset --drive_root {DRIVE_ROOT} --scan_json {AUDIT}/exif_scan.json --vitextvqa_images {IMG_TV} --rot_dir {ROT_DIR} --out {OUT}
SUB = f'{OUT}/test_exif_subset.json'
if os.path.exists(f'{EXP_ROOT}/{ROT_EVAL}/metrics.json'):
    print('⏭ eval ảnh xoay đã có -> bỏ qua')
else:
    run_cmd(f'python -u -m src.eval.run_baseline --dataset {DS} --data_file {SUB} --run_name {ROT_EVAL} '
            f'--chunk {CHUNK} --adapter {AD_B3F} {OCR_ARGS} --image_dir {ROT_DIR}')
mr = json.load(open(f'{EXP_ROOT}/{ROT_EVAL}/metrics.json', encoding='utf-8'))
print(f"✔ ảnh xoay: EM={mr['metrics']['exact_match']:.4f} ANLS={mr['metrics']['anls']:.4f} (n={len(json.load(open(SUB)))})")

## 6. So ghép cặp + kiểm V5

- **V5 (hợp lệ):** ≥ 99,5% câu có ảnh **không** cờ xoay cho dự đoán **trùng** run W10. Không đạt → run không dùng được, báo Claude.
- **Báo cáo (L2):** full test và riêng 126 câu: EM, ANLS, ΔEM (McNemar, bootstrap CI95), ΔANLS (bootstrap CI95); B2 → B3-full cũ / mới.
- **Chỉ mô tả:** ảnh xoay so với ảnh chưa xoay trên 126 câu (cùng OCR đã sửa).

*Kỳ vọng: `V5: … trùng ✅`, rồi bảng. Khoảng 3–5 phút (bootstrap).*

In [ ]:
run_cmd(f"python -W ignore -m src.data.exif_audit --compare --drive_root {DRIVE_ROOT} --scan_json {AUDIT}/exif_scan.json "
        f"--old_run {EXP_ROOT}/{REF['B3full_W10']} --new_run {EXP_ROOT}/{NEW_EVAL} --b2_run {EXP_ROOT}/{REF['B2_W07']} "
        f"--rot_run {EXP_ROOT}/{ROT_EVAL} --out {OUT} > /dev/null")
S = json.load(open(f'{OUT}/exif_fix_b3full_summary.json', encoding='utf-8'))
v5 = S['V5_unaffected_identical']
print(f"V5: {v5['n_identical']}/{S['n_unaffected']} câu không bị ảnh hưởng trùng W10 ({v5['pct']}%) {'✅' if v5['pass'] else '❌'}")
assert v5['pass'], '❌ V5 không đạt -> run không hợp lệ, gửi Claude output mục 7'
for k, lab in (('full_test', 'full test'), ('affected_only', '126 câu ảnh cờ xoay')):
    d = S[k]
    print(f"{lab:22s} n={d['n']:5d} | EM {d['em_a']} → {d['em_b']} (Δ {d['d_em']}, CI {d.get('d_em_ci95')}, p={d['mcnemar_p']:.2g}) "
          f"| ANLS {d['anls_a']} → {d['anls_b']} (Δ {d['d_anls']}, CI {d.get('d_anls_ci95')})")
for k in ('B2_to_B3full_old', 'B2_to_B3full_new'):
    d = S[k]
    print(f"{k:18s} ΔEM {d['d_em']} ({d['b_only']}/{d['a_only']}, p={d['mcnemar_p']:.2g})")
d = S['descriptive_rotated_image_vs_new']
print(f"ảnh xoay vs chưa xoay (126 câu, chỉ mô tả): EM {d['em_a']} → {d['em_b']} (Δ {d['d_em']}, p={d['mcnemar_p']:.2g})")

## 7. ⭐ Khối gửi Claude

*Kỳ vọng: khoảng 10 dòng, bắt đầu bằng `=== R-EXIF bước 2B ===`.*

In [ ]:
print('=== R-EXIF bước 2B ===')
!git log -1 --format='commit %h'
print('GPU', GPU)
print(json.dumps({k: S[k] for k in ('n_questions', 'n_affected', 'n_unaffected', 'V5_unaffected_identical', 'verdict')}, ensure_ascii=False))
for k in ('full_test', 'affected_only', 'unaffected_only', 'B2_to_B3full_old', 'B2_to_B3full_new', 'descriptive_rotated_image_vs_new'):
    print(k, json.dumps(S[k], ensure_ascii=False))
print('metrics mới', json.dumps(S.get('metrics_json_new'), ensure_ascii=False))
print('metrics W10', json.dumps(S.get('metrics_json_old'), ensure_ascii=False))

## 8. Trả máy

Kết quả: run `W14_B3full_eval_exiffix_…` trên Drive `experiments/`, tổng hợp ở `experiments/W14_exif_fix/exif_fix_b3full_summary.json`.

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')